# EV Battery Range Prediction
**Predicting remaining driving range of electric vehicles using machine learning**

Relevant to BMW Group's iX / i-Series electromobility division.  
Range anxiety is one of the top concerns for EV drivers. A reliable range predictor — trained on real driving conditions — directly improves the customer experience.

**Goal:** Predict remaining range (km) from real-time driving parameters using regression models.

---

## 1. Setup & Synthetic Dataset Generation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print('Libraries loaded.')

In [ ]:
# ---------------------------------------------------------------
# Synthetic EV driving dataset
# Each row = one driving session snapshot
# ---------------------------------------------------------------
N = 2000

battery_soc        = np.random.uniform(5, 100, N)          # State of Charge (%)
avg_speed          = np.random.uniform(10, 160, N)          # km/h
ambient_temp       = np.random.uniform(-15, 40, N)          # Celsius
battery_temp       = ambient_temp + np.random.uniform(2, 15, N)
acceleration_score = np.random.uniform(0, 1, N)             # 0=gentle, 1=aggressive
hvac_load          = np.random.uniform(0, 5, N)             # kW (heating/cooling)
payload_kg         = np.random.uniform(0, 600, N)           # passengers + luggage
road_gradient      = np.random.uniform(-8, 8, N)            # % slope
tire_pressure_loss = np.random.uniform(0, 0.3, N)           # normalized pressure deficit

# Physics-inspired range formula (simplified)
range_km = (
    battery_soc * 4.5                          # capacity effect
    - avg_speed * 0.8                          # speed increases consumption
    - np.abs(ambient_temp - 20) * 2.0          # thermal penalty
    - acceleration_score * 80                  # aggressive driving
    - hvac_load * 15                           # climate control
    - payload_kg * 0.05                        # weight
    - road_gradient * 3                        # gradient
    - tire_pressure_loss * 40                  # rolling resistance
    + np.random.normal(0, 8, N)               # noise
)
range_km = np.clip(range_km, 0, 600)

df = pd.DataFrame({
    'battery_soc_%':       battery_soc,
    'avg_speed_kmh':       avg_speed,
    'ambient_temp_c':      ambient_temp,
    'battery_temp_c':      battery_temp,
    'acceleration_score':  acceleration_score,
    'hvac_load_kw':        hvac_load,
    'payload_kg':          payload_kg,
    'road_gradient_%':     road_gradient,
    'tire_pressure_loss':  tire_pressure_loss,
    'remaining_range_km':  range_km
})

print(f'Dataset: {df.shape[0]} samples, {df.shape[1]} features')
df.head()

## 2. Exploratory Data Analysis

In [ ]:
print('=== Dataset Summary ===')
print(df.describe().round(2))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0,0].scatter(df['battery_soc_%'], df['remaining_range_km'], alpha=0.3, color='steelblue', s=10)
axes[0,0].set_xlabel('Battery SoC (%)')
axes[0,0].set_ylabel('Remaining Range (km)')
axes[0,0].set_title('SoC vs Range')

axes[0,1].scatter(df['avg_speed_kmh'], df['remaining_range_km'], alpha=0.3, color='darkorange', s=10)
axes[0,1].set_xlabel('Average Speed (km/h)')
axes[0,1].set_ylabel('Remaining Range (km)')
axes[0,1].set_title('Speed vs Range')

axes[1,0].scatter(df['ambient_temp_c'], df['remaining_range_km'], alpha=0.3, color='seagreen', s=10)
axes[1,0].set_xlabel('Ambient Temperature (°C)')
axes[1,0].set_ylabel('Remaining Range (km)')
axes[1,0].set_title('Temperature vs Range (Thermal Penalty)')

axes[1,1].hist(df['remaining_range_km'], bins=40, color='mediumpurple', edgecolor='white')
axes[1,1].set_xlabel('Remaining Range (km)')
axes[1,1].set_title('Range Distribution')

plt.tight_layout()
plt.suptitle('EV Range Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.savefig('eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: eda_plots.png')

In [ ]:
plt.figure(figsize=(10, 6))
corr = df.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, linewidths=0.5)
plt.title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Model Training

In [ ]:
FEATURES = [c for c in df.columns if c != 'remaining_range_km']
TARGET   = 'remaining_range_km'

X = df[FEATURES].values
y = df[TARGET].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

models = {
    'Linear Regression':      LinearRegression(),
    'Random Forest':          RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting':      GradientBoostingRegressor(n_estimators=100, random_state=42),
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)
    results.append({'Model': name, 'RMSE (km)': round(rmse, 2), 'MAE (km)': round(mae, 2), 'R²': round(r2, 4)})
    print(f'{name:25s} | RMSE: {rmse:.2f} km | MAE: {mae:.2f} km | R²: {r2:.4f}')

results_df = pd.DataFrame(results)
print()
print(results_df.to_string(index=False))

## 4. Feature Importance & Predictions

In [ ]:
rf_model = models['Random Forest']
importances = pd.Series(rf_model.feature_importances_, index=FEATURES).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

importances.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Feature Importance (Random Forest)', fontweight='bold')
axes[0].set_xlabel('Importance Score')

y_pred_rf = models['Random Forest'].predict(X_test)
axes[1].scatter(y_test, y_pred_rf, alpha=0.4, color='steelblue', s=15)
axes[1].plot([0, 600], [0, 600], 'r--', lw=2, label='Perfect prediction')
axes[1].set_xlabel('Actual Range (km)')
axes[1].set_ylabel('Predicted Range (km)')
axes[1].set_title(f'Random Forest: Predicted vs Actual (R²={r2_score(y_test, y_pred_rf):.3f})', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('model_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Results Summary

In [ ]:
best = results_df.loc[results_df['R²'].idxmax()]
print('=== RESULTS SUMMARY ===')
print(results_df.to_string(index=False))
print()
print(f'Best model: {best["Model"]} (R² = {best["R²"]}, MAE = {best["MAE (km)"]} km)')
print()
print('Key insights:')
print('  - Battery SoC is the dominant predictor of remaining range')
print('  - Ambient temperature penalty is significant (cold climate range loss)')
print('  - Aggressive driving (acceleration score) reduces range by up to 80 km')
print('  - HVAC load creates meaningful range reduction in winter conditions')
print()
print('BMW relevance:')
print('  - Foundation for real-time range estimation in BMW iX / i4 / i5 software stack')
print('  - Can be extended with GPS elevation data, traffic density, driver profile')
print('  - Explainable predictions support customer trust (responsible AI)')